In [ ]:
import mlflow
import pandas as pd

print("MLflow versão:", mlflow.__version__)

# Tracking local via SQLite (arquivo mlflow.db no repo, ainda sem servidor remoto)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("datathon-bandit")

# Reload dos dados (mesma lógica das Etapas 3/4, pra o notebook ficar independente)
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
arm_train = pd.read_csv("../data/processed/arm_train.csv").squeeze()
arm_test = pd.read_csv("../data/processed/arm_test.csv").squeeze()

print(X_train.shape, X_test.shape)
print(arm_train.value_counts())

MLflow versão: 3.16.0


2026/09/13 19:24:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/13 19:24:24 INFO mlflow.store.db.utils: Updating database tables
2026/09/13 19:24:24 INFO mlflow.tracking.fluent: Experiment with name 'datathon-bandit' does not exist. Creating a new experiment.


(32950, 51) (8238, 51)
arm
0    20908
1    12042
Name: count, dtype: int64


In [4]:
from sklearn.linear_model import LogisticRegression

SEED = 42
DATASET_VERSION = "bank-additional-full_v1"  # mesmo dataset usado desde a Etapa 1

# Máscaras por braço
mask_arm0_train = arm_train == 0
mask_arm1_train = arm_train == 1

model_arm0 = LogisticRegression(max_iter=1000, random_state=SEED)
model_arm0.fit(X_train[mask_arm0_train], y_train[mask_arm0_train])

model_arm1 = LogisticRegression(max_iter=1000, random_state=SEED)
model_arm1.fit(X_train[mask_arm1_train], y_train[mask_arm1_train])

print("Linhas treino - braço 0:", mask_arm0_train.sum())
print("Linhas treino - braço 1:", mask_arm1_train.sum())

# Baseline: sempre recomendar braço 0 (cellular) -> conversão real do braço 0 no teste
mask_arm0_test = arm_test == 0
baseline_conversion = y_test[mask_arm0_test].mean()
print(f"Baseline (sempre braço 0) - conversão no teste: {baseline_conversion:.4f}")

# Registrar run do baseline no MLflow
with mlflow.start_run(run_name="baseline_sempre_cellular"):
    mlflow.log_param("policy", "baseline")
    mlflow.log_param("seed", SEED)
    mlflow.log_param("dataset_version", DATASET_VERSION)
    mlflow.log_metric("conversion_rate", baseline_conversion)
    mlflow.log_metric("coverage", 1.0)  # baseline se aplica a 100% dos casos

print("Run do baseline registrada no MLflow.")

/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Linhas treino - braço 0: 20908
Linhas treino - braço 1: 12042
Baseline (sempre braço 0) - conversão no teste: 0.1488
Run do baseline registrada no MLflow.


/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
import numpy as np

epsilons = [0.10, 0.05, 0.01]

# Probabilidades previstas de conversão em cada braço, para todo o conjunto de teste
prob_arm0_test = model_arm0.predict_proba(X_test)[:, 1]
prob_arm1_test = model_arm1.predict_proba(X_test)[:, 1]

for eps in epsilons:
    rng = np.random.RandomState(SEED)
    n = len(X_test)
    explore = rng.rand(n) < eps
    exploit_choice = (prob_arm1_test > prob_arm0_test).astype(int)
    random_choice = rng.randint(0, 2, size=n)
    chosen_arm = np.where(explore, random_choice, exploit_choice)

    # Replay: só conta os casos em que a escolha do bandit bateu com o braço real usado no histórico
    matches = chosen_arm == arm_test.values
    coverage = matches.mean()
    conversion = y_test.values[matches].mean()
    lift = conversion - baseline_conversion

    print(f"epsilon={eps:.2f} | cobertura={coverage:.4f} | conversão={conversion:.4f} | lift={lift:+.4f}")

    with mlflow.start_run(run_name=f"epsilon_greedy_{eps}"):
        mlflow.log_param("policy", "epsilon_greedy")
        mlflow.log_param("epsilon", eps)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("dataset_version", DATASET_VERSION)
        mlflow.log_metric("coverage", coverage)
        mlflow.log_metric("conversion_rate", conversion)
        mlflow.log_metric("lift_vs_baseline", lift)

print("3 runs do Epsilon-Greedy registradas no MLflow.")

epsilon=0.10 | cobertura=0.5392 | conversão=0.1371 | lift=-0.0117
epsilon=0.05 | cobertura=0.5435 | conversão=0.1403 | lift=-0.0085
epsilon=0.01 | cobertura=0.5455 | conversão=0.1415 | lift=-0.0073
3 runs do Epsilon-Greedy registradas no MLflow.
